In [27]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
import os

In [28]:
df = pd.read_csv('../data/processed/telco_features.csv')
print(df.shape)

# Make Churn the last column on the df
cols = [c for c in df.columns if c != 'Churn'] + ['Churn']
df = df[cols]

print(df.columns[-1])

(7032, 30)
Churn


In [29]:
X = df[[ c for c in df.columns if c != 'Churn']]
y = df['Churn']


X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f'X_train shape: {X_train.shape}')
print(f'X_test shape: {X_test.shape}')
print('--------------------')
print(f'y_train mean: {y_train.mean()}')
print(f'y_test mean: {y_test.mean()}')



X_train shape: (5625, 29)
X_test shape: (1407, 29)
--------------------
y_train mean: 0.2657777777777778
y_test mean: 0.2658137882018479


In [30]:
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)

y_pred_proba = lr.predict_proba(X_test)[:, 1]

score = roc_auc_score(y_test, y_pred_proba)
print(f'roc_auc_score: {score}')

roc_auc_score: 0.8347849832531797


C:\Users\user\AppData\Roaming\Python\Python313\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [31]:
y_pred = lr.predict(X_test)

class_report = classification_report(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)

print(class_report)
print(conf_matrix)

              precision    recall  f1-score   support

           0       0.84      0.89      0.87      1033
           1       0.64      0.53      0.58       374

    accuracy                           0.80      1407
   macro avg       0.74      0.71      0.72      1407
weighted avg       0.79      0.80      0.79      1407

[[920 113]
 [174 200]]


In [32]:
print(X_train.describe().T[['mean', 'std', 'min', 'max']])

                                                mean          std     min  \
gender                                      0.501867     0.500041   0.000   
SeniorCitizen                               0.161778     0.368280   0.000   
Partner                                     0.486044     0.499850   0.000   
Dependents                                  0.299556     0.458104   0.000   
tenure                                     32.562311    24.542421   1.000   
PhoneService                                0.903111     0.295833   0.000   
MultipleLines                               0.424000     0.494234   0.000   
OnlineSecurity                              0.286933     0.452370   0.000   
OnlineBackup                                0.350400     0.477138   0.000   
DeviceProtection                            0.345600     0.475606   0.000   
TechSupport                                 0.294756     0.455973   0.000   
StreamingTV                                 0.388978     0.487562   0.000   

In [33]:
coef_series = pd.Series(
    lr.coef_[0],
    index=X_train.columns
).sort_values()

print("Bottom 10 coefficients:")
print(coef_series.head(10))

print("\nTop 10 coefficients:")
print(coef_series.tail(10))

Bottom 10 coefficients:
Contract_Two year                         -0.876401
PhoneService                              -0.859177
InternetService_No                        -0.827286
OnlineSecurity                            -0.460260
TechSupport                               -0.416664
PaymentMethod_Bank transfer (automatic)   -0.255701
OnlineBackup                              -0.233556
Dependents                                -0.223933
InternetService_DSL                       -0.204383
PaymentMethod_Credit card (automatic)     -0.203031
dtype: float64

Top 10 coefficients:
HasProtectionBundle               0.054601
PaymentMethod_Electronic check    0.120180
StreamingMovies                   0.160558
SeniorCitizen                     0.184500
StreamingTV                       0.197058
PaperlessBilling                  0.270997
MultipleLines                     0.311657
IsNewCustomer                     0.496984
InternetService_Fiber optic       0.512616
Contract_Month-to-month         

In [34]:
import joblib

os.makedirs('../models', exist_ok=True)
joblib.dump(lr, '../models/baseline_logreg.joblib')
print('Saved to ../models/baseline_logreg.joblib')
print(f'File size: {os.path.getsize("../models/baseline_logreg.joblib"):,} bytes')

Saved to ../models/baseline_logreg.joblib
File size: 1,919 bytes


## Baseline Model Summary

**Model:** Logistic Regression (max_iter=1000, random_state=42)
**Split:** 80/20 train/test, stratified on Churn (train churn rate: 26.58%, test: 26.58%)

| Metric | Score |
|--------|-------|
| ROC-AUC | 0.8348 |
| Accuracy | 0.80 |
| Precision (Churn=1) | 0.64 |
| Recall (Churn=1) | 0.53 |
| F1 (Churn=1) | 0.58 |

**Confusion matrix:**
|  | Predicted No | Predicted Yes |
|---|---|---|
| Actual No | 920 | 113 |
| Actual Yes | 174 | 200 |

**Key observations:**
- ROC-AUC of 0.8348 is a strong, credible baseline — in line with typical published benchmarks for this dataset
- At the default 0.5 threshold, the model misses 47% of actual churners (174 false negatives) — a direct consequence of class imbalance, not a flaw in the model itself. Threshold tuning is a candidate improvement for later notebooks
- Model failed to fully converge (`ConvergenceWarning`) — feature scale diagnostic confirmed why: TotalCharges (std ≈ 2,276) sits three orders of magnitude above binary flags (std ≈ 0.5)
- Coefficient directions agree with EDA and feature-engineering correlations: Contract_Month-to-month (+0.55) and InternetService_Fiber optic (+0.51) are the strongest churn-increasing signals; Contract_Two year (-0.88) is the strongest churn-decreasing signal
- IsNewCustomer (+0.50) ranks among the top positive coefficients, consistent with its 0.320 correlation found in feature engineering
- Some coefficients (PhoneService: -0.86, InternetService_No: -0.83) appear inflated relative to their earlier correlation signal — likely a scale artifact rather than genuine importance, reinforcing the case for scaling

**Model saved:** `models/baseline_logreg.joblib` (1,919 bytes)

### Next step
Model improvement (06_model_improvement.ipynb) — apply feature scaling, try LightGBM/Random Forest, and consider threshold tuning to improve recall on churners without sacrificing ROC-AUC.